### Dicionário de Dados (`flights.csv`)

Abaixo estão as descrições das variáveis contidas no conjunto de dados de voos, incluindo seus tipos e unidades de medida.

| Coluna | Descrição | Tipo / Unidade |
| :--- | :--- | :--- |
| **YEAR** | Ano do voo (ex.: 2015) | Inteiro |
| **MONTH** | Mês do voo (1 a 12) | Inteiro |
| **DAY** | Dia do mês do voo (1 a 31) | Inteiro |
| **DAY_OF_WEEK** | Dia da semana (1 = Segunda, 7 = Domingo) | Inteiro |
| **AIRLINE** | Código da companhia aérea (ex.: AA = American Airlines) | Categórica |
| **FLIGHT_NUMBER** | Número do voo | Inteiro |
| **TAIL_NUMBER** | Número de registro da aeronave | Texto |
| **ORIGIN_AIRPORT** | Código IATA do aeroporto de origem (ex.: ATL) | Categórica |
| **DESTINATION_AIRPORT** | Código IATA do aeroporto de destino | Categórica |
| **SCHEDULED_DEPARTURE** | Horário de partida programado | HHMM (Inteiro) |
| **DEPARTURE_TIME** | Horário real de partida | HHMM (Inteiro) |
| **DEPARTURE_DELAY** | Atraso na partida | Numérico (Minutos) |
| **TAXI_OUT** | Tempo gasto taxiando até a decolagem | Numérico (Minutos) |
| **WHEELS_OFF** | Horário em que o avião decolou | HHMM (Inteiro) |
| **SCHEDULED_TIME** | Tempo total programado de voo | Numérico (Minutos) |
| **ELAPSED_TIME** | Tempo total real de voo | Numérico (Minutos) |
| **AIR_TIME** | Tempo no ar | Numérico (Minutos) |
| **DISTANCE** | Distância entre origem e destino | Numérico (Milhas) |
| **WHEELS_ON** | Horário em que as rodas tocaram o solo | HHMM (Inteiro) |
| **TAXI_IN** | Tempo taxiando até o portão de desembarque | Numérico (Minutos) |
| **SCHEDULED_ARRIVAL** | Horário de chegada programado | HHMM (Inteiro) |
| **ARRIVAL_TIME** | Horário de chegada real | HHMM (Inteiro) |
| **ARRIVAL_DELAY** | Atraso na chegada | Numérico (Minutos) |
| **DIVERTED** | Indica se o voo foi desviado (1 = Sim, 0 = Não) | Binária |
| **CANCELLED** | Indica se o voo foi cancelado (1 = Sim, 0 = Não) | Binária |
| **CANCELLATION_REASON** | Motivo do cancelamento (A=Airline, B=Weather, C=NAS, D=Security) | Categórica |
| **AIR_SYSTEM_DELAY** | Atraso causado por controle de tráfego aéreo | Numérico (Minutos) |
| **SECURITY_DELAY** | Atraso causado por problemas de segurança | Numérico (Minutos) |
| **AIRLINE_DELAY** | Atraso causado pela companhia aérea | Numérico (Minutos) |
| **LATE_AIRCRAFT_DELAY** | Atraso causado por chegada tardia da aeronave | Numérico (Minutos) |
| **WEATHER_DELAY** | Atraso causado por condições meteorológicas | Numérico (Minutos) |

In [2]:
#importando bibliotecas
import pandas as pd
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff 
from plotly.subplots import make_subplots

import pandas as pd
from sklearn.cluster import KMeans
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score

pd.set_option('display.max_columns', None)

In [3]:
#importando dados
df_flights = pd.read_csv('../data/flights.csv')
df_airlines = pd.read_csv('../data/airlines.csv')
df_airports = pd.read_csv('../data/airports.csv')

C:\Users\leand\AppData\Local\Temp\ipykernel_26596\761702122.py:2: DtypeWarning: Columns (0: ORIGIN_AIRPORT, 1: DESTINATION_AIRPORT) have mixed types. Specify dtype option on import or set low_memory=False.
  df_flights = pd.read_csv('../data/flights.csv')


### Tratamento de dados

In [4]:
#renomenado colunas
df_airlines = df_airlines.rename(columns={'AIRLINE': 'AIRLINE_NAME'})
df_airport_origin = df_airports.rename(columns={
    'IATA_CODE': 'ID',
    'AIRPORT': 'ORIGIN_AIRPORT_NAME',
    'CITY': 'ORIGIN_CITY',
    'STATE': 'ORIGIN_STATE',
    'COUNTRY': 'ORIGIN_COUNTRY',
    'LATITUDE': 'ORIGIN_LATITUDE',
    'LONGITUDE': 'ORIGIN_LONGITUDE'
})

df_airport_dest = df_airports.rename(columns={
    'IATA_CODE': 'ID',
    'AIRPORT': 'DEST_AIRPORT_NAME',
    'CITY': 'DEST_CITY',
    'STATE': 'DEST_STATE',
    'COUNTRY': 'DEST_COUNTRY',
    'LATITUDE': 'DEST_LATITUDE',
    'LONGITUDE': 'DEST_LONGITUDE'
})

df = (
    df_flights
    .merge(df_airlines, left_on='AIRLINE', right_on='IATA_CODE', how='left')
    .drop(columns=['IATA_CODE'])
    
    .merge(df_airport_origin, left_on='ORIGIN_AIRPORT', right_on='ID', how='left')
    .drop(columns=['ID'])
    
    .merge(df_airport_dest, left_on='DESTINATION_AIRPORT', right_on='ID', how='left')
    .drop(columns=['ID'])
)

df.head()

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,TAXI_OUT,WHEELS_OFF,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,WHEELS_ON,TAXI_IN,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY,AIRLINE_NAME,ORIGIN_AIRPORT_NAME,ORIGIN_CITY,ORIGIN_STATE,ORIGIN_COUNTRY,ORIGIN_LATITUDE,ORIGIN_LONGITUDE,DEST_AIRPORT_NAME,DEST_CITY,DEST_STATE,DEST_COUNTRY,DEST_LATITUDE,DEST_LONGITUDE
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,2354.0,-11.0,21.0,15.0,205.0,194.0,169.0,1448,404.0,4.0,430,408.0,-22.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,Alaska Airlines Inc.,Ted Stevens Anchorage International Airport,Anchorage,AK,USA,61.17432,-149.99619,Seattle-Tacoma International Airport,Seattle,WA,USA,47.44898,-122.30931
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,2.0,-8.0,12.0,14.0,280.0,279.0,263.0,2330,737.0,4.0,750,741.0,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,American Airlines Inc.,Los Angeles International Airport,Los Angeles,CA,USA,33.94254,-118.40807,Palm Beach International Airport,West Palm Beach,FL,USA,26.68316,-80.09559
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,18.0,-2.0,16.0,34.0,286.0,293.0,266.0,2296,800.0,11.0,806,811.0,5.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,US Airways Inc.,San Francisco International Airport,San Francisco,CA,USA,37.61900,-122.37484,Charlotte Douglas International Airport,Charlotte,NC,USA,35.21401,-80.94313
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,15.0,-5.0,15.0,30.0,285.0,281.0,258.0,2342,748.0,8.0,805,756.0,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,American Airlines Inc.,Los Angeles International Airport,Los Angeles,CA,USA,33.94254,-118.40807,Miami International Airport,Miami,FL,USA,25.79325,-80.29056
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,24.0,-1.0,11.0,35.0,235.0,215.0,199.0,1448,254.0,5.0,320,259.0,-21.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,Alaska Airlines Inc.,Seattle-Tacoma International Airport,Seattle,WA,USA,47.44898,-122.30931,Ted Stevens Anchorage International Airport,Anchorage,AK,USA,61.17432,-149.99619


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5819079 entries, 0 to 5819078
Data columns (total 44 columns):
 #   Column               Dtype  
---  ------               -----  
 0   YEAR                 int64  
 1   MONTH                int64  
 2   DAY                  int64  
 3   DAY_OF_WEEK          int64  
 4   AIRLINE              str    
 5   FLIGHT_NUMBER        int64  
 6   TAIL_NUMBER          str    
 7   ORIGIN_AIRPORT       object 
 8   DESTINATION_AIRPORT  object 
 9   SCHEDULED_DEPARTURE  int64  
 10  DEPARTURE_TIME       float64
 11  DEPARTURE_DELAY      float64
 12  TAXI_OUT             float64
 13  WHEELS_OFF           float64
 14  SCHEDULED_TIME       float64
 15  ELAPSED_TIME         float64
 16  AIR_TIME             float64
 17  DISTANCE             int64  
 18  WHEELS_ON            float64
 19  TAXI_IN              float64
 20  SCHEDULED_ARRIVAL    int64  
 21  ARRIVAL_TIME         float64
 22  ARRIVAL_DELAY        float64
 23  DIVERTED             int64  
 24  CANCELLED

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5819079 entries, 0 to 5819078
Data columns (total 44 columns):
 #   Column               Dtype  
---  ------               -----  
 0   YEAR                 int64  
 1   MONTH                int64  
 2   DAY                  int64  
 3   DAY_OF_WEEK          int64  
 4   AIRLINE              str    
 5   FLIGHT_NUMBER        int64  
 6   TAIL_NUMBER          str    
 7   ORIGIN_AIRPORT       object 
 8   DESTINATION_AIRPORT  object 
 9   SCHEDULED_DEPARTURE  int64  
 10  DEPARTURE_TIME       float64
 11  DEPARTURE_DELAY      float64
 12  TAXI_OUT             float64
 13  WHEELS_OFF           float64
 14  SCHEDULED_TIME       float64
 15  ELAPSED_TIME         float64
 16  AIR_TIME             float64
 17  DISTANCE             int64  
 18  WHEELS_ON            float64
 19  TAXI_IN              float64
 20  SCHEDULED_ARRIVAL    int64  
 21  ARRIVAL_TIME         float64
 22  ARRIVAL_DELAY        float64
 23  DIVERTED             int64  
 24  CANCELLED

In [7]:
df.shape

(5819079, 44)

Tratamento de nulos

In [8]:
nulos = pd.DataFrame({
    'Nulos': df.isnull().sum(),
    '%': (df.isnull().sum() / len(df) * 100).round(2)
})

print(nulos[nulos['Nulos'] > 0])

                       Nulos      %
TAIL_NUMBER            14721   0.25
DEPARTURE_TIME         86153   1.48
DEPARTURE_DELAY        86153   1.48
TAXI_OUT               89047   1.53
WHEELS_OFF             89047   1.53
SCHEDULED_TIME             6   0.00
ELAPSED_TIME          105071   1.81
AIR_TIME              105071   1.81
WHEELS_ON              92513   1.59
TAXI_IN                92513   1.59
ARRIVAL_TIME           92513   1.59
ARRIVAL_DELAY         105071   1.81
CANCELLATION_REASON  5729195  98.46
AIR_SYSTEM_DELAY     4755640  81.72
SECURITY_DELAY       4755640  81.72
AIRLINE_DELAY        4755640  81.72
LATE_AIRCRAFT_DELAY  4755640  81.72
WEATHER_DELAY        4755640  81.72
ORIGIN_AIRPORT_NAME   486165   8.35
ORIGIN_CITY           486165   8.35
ORIGIN_STATE          486165   8.35
ORIGIN_COUNTRY        486165   8.35
ORIGIN_LATITUDE       490770   8.43
ORIGIN_LONGITUDE      490770   8.43
DEST_AIRPORT_NAME     486165   8.35
DEST_CITY             486165   8.35
DEST_STATE            486165

In [9]:
#preenchendo campos nulos que ocorrem por não haver atraso
delay_cols = ['AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY', 'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY']
df[delay_cols] = df[delay_cols].fillna(0)

#removendo motivo de cancelamento, praticamente vazia
df = df.drop(columns=['CANCELLATION_REASON'])

#removendo demais nulos
df = df.dropna(subset=[
    'ARRIVAL_DELAY', 
    'ORIGIN_CITY', 
    'DEST_CITY', 
    'ORIGIN_LATITUDE', 
    'DEST_LATITUDE',
    'TAIL_NUMBER'
])

print(df.isnull().sum().sum())

0


Filtrando dados

In [10]:
#removendo voos cancelados e desviados, não ajudam na classificação
df = df[(df['CANCELLED'] == 0) & (df['DIVERTED'] == 0)]


Transformando dados

In [11]:
#criando a variável target
df['IS_DELAYED'] = (df['ARRIVAL_DELAY'] > 15).astype(int)

### EDA

In [12]:
#estatísticas das colunas numéricas
display(df.describe(include=[float, int]))

,YEAR,MONTH,DAY,DAY_OF_WEEK,FLIGHT_NUMBER,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,TAXI_OUT,WHEELS_OFF,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,WHEELS_ON,TAXI_IN,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY,ORIGIN_LATITUDE,ORIGIN_LONGITUDE,DEST_LATITUDE,DEST_LONGITUDE,IS_DELAYED
count,5222000.0,5.222000e+06,5.222000e+06,5.222000e+06,5.222000e+06,5.222000e+06,5.222000e+06,5.222000e+06,5.222000e+06,5.222000e+06,5.222000e+06,5.222000e+06,5.222000e+06,5.222000e+06,5.222000e+06,5.222000e+06,5.222000e+06,5.222000e+06,5.222000e+06,5222000.0,5222000.0,5.222000e+06,5.222000e+06,5.222000e+06,5.222000e+06,5.222000e+06,5.222000e+06,5.222000e+06,5.222000e+06,5.222000e+06,5.222000e+06
mean,2015.0,6.228877e+00,1.569435e+01,3.925464e+00,2.169281e+03,1.329431e+03,1.335965e+03,9.700099e+00,1.610082e+01,1.357885e+03,1.420873e+02,1.372773e+02,1.137419e+02,8.257542e+02,1.470982e+03,7.434598e+00,1.493247e+03,1.475903e+03,4.890091e+00,0.0,0.0,2.588242e+00,1.491421e-02,3.627041e+00,4.544785e+00,5.712951e-01,3.662658e+01,-9.558052e+01,3.662642e+01,-9.558102e+01,1.847708e-01
std,0.0,3.377674e+00,8.765640e+00,1.990362e+00,1.758518e+03,4.840966e+02,4.974246e+02,3.739691e+01,8.978905e+00,4.990809e+02,7.538905e+01,7.432545e+01,7.232390e+01,6.089703e+02,5.234858e+02,5.668309e+00,5.076978e+02,5.276576e+02,3.978903e+01,0.0,0.0,1.339577e+01,9.559389e-01,2.224489e+01,2.114842e+01,9.096096e+00,6.009523e+00,1.819463e+01,6.011540e+00,1.819930e+01,3.881115e-01
min,2015.0,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,-8.200000e+01,1.000000e+00,1.000000e+00,1.800000e+01,1.400000e+01,7.000000e+00,3.100000e+01,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,-8.700000e+01,0.0,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.348345e+01,-1.766460e+02,1.348345e+01,-1.766460e+02,0.000000e+00
25%,2015.0,3.000000e+00,8.000000e+00,2.000000e+00,7.280000e+02,9.160000e+02,9.210000e+02,-5.000000e+00,1.100000e+01,9.350000e+02,8.600000e+01,8.200000e+01,6.100000e+01,3.730000e+02,1.054000e+03,4.000000e+00,1.110000e+03,1.058000e+03,-1.300000e+01,0.0,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,3.289595e+01,-1.119778e+02,3.289595e+01,-1.119778e+02,0.000000e+00
50%,2015.0,6.000000e+00,1.600000e+01,4.000000e+00,1.680000e+03,1.325000e+03,1.331000e+03,-1.000000e+00,1.400000e+01,1.343000e+03,1.230000e+02,1.190000e+02,9.500000e+01,6.510000e+02,1.508000e+03,6.000000e+00,1.520000e+03,1.512000e+03,-5.000000e+00,0.0,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,3.724433e+01,-9.025803e+01,3.724433e+01,-9.025803e+01,0.000000e+00
75%,2015.0,9.000000e+00,2.300000e+01,6.000000e+00,3.230000e+03,1.730000e+03,1.741000e+03,8.000000e+00,1.900000e+01,1.755000e+03,1.740000e+02,1.690000e+02,1.440000e+02,1.067000e+03,1.912000e+03,9.000000e+00,1.918000e+03,1.917000e+03,8.000000e+00,0.0,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,4.078839e+01,-8.168786e+01,4.078839e+01,-8.168786e+01,0.000000e+00
max,2015.0,1.200000e+01,3.100000e+01,7.000000e+00,9.320000e+03,2.359000e+03,2.400000e+03,1.988000e+03,2.250000e+02,2.400000e+03,7.180000e+02,7.660000e+02,6.900000e+02,4.983000e+03,2.400000e+03,2.480000e+02,2.400000e+03,2.400000e+03,1.971000e+03,0.0,0.0,1.134000e+03,5.730000e+02,1.971000e+03,1.331000e+03,1.211000e+03,7.128545e+01,-6.479856e+01,7.128545e+01,-6.479856e+01,1.000000e+00


Qual é a distribuição na base entre voos atrasados e pontuais?

In [13]:
df_balance = df['IS_DELAYED'].value_counts().reset_index()
df_balance.columns = ['TARGET_STATUS', 'COUNT']
df_balance['STATUS_NAME'] = df_balance['TARGET_STATUS'].map({0: 'Pontual', 1: 'Atrasado'})

fig = px.pie(
    df_balance, 
    values='COUNT', 
    names='STATUS_NAME', 
    title='Distribuição dos Voos',
    color='STATUS_NAME',
    color_discrete_map={'Pontual': '#3498db', 'Atrasado': '#e74c3c'},
    hole=0.4
)
fig.update_traces(
    textinfo='percent+label',
    pull=[0, 0.1]
)
fig.update_layout(
    template='plotly_white',
    height=600, 
)
fig.show()

Quais são as principais causas de atraso?

In [14]:
#somando todos os minutos de atraso por categoria
causes = {
    'Sistema Aéreo': df['AIR_SYSTEM_DELAY'].sum(),
    'Segurança': df['SECURITY_DELAY'].sum(),
    'Companhia Aérea': df['AIRLINE_DELAY'].sum(),
    'Aeronave Tardia': df['LATE_AIRCRAFT_DELAY'].sum(),
    'Clima': df['WEATHER_DELAY'].sum()
}

df_causes = pd.DataFrame(list(causes.items()), columns=['Causa', 'Total Minutos'])

fig = px.bar(
    df_causes.sort_values('Total Minutos', ascending=True), 
    x='Total Minutos', 
    y='Causa', 
    orientation='h',
    title='Principais Causas de Atrasos por Minutos Acumulados',
    color='Causa'
)

fig.update_layout(showlegend=False, template='plotly_white')
fig.show()

A análise das causas revela que a eficiência operacional é o maior desafio do setor. O predominínio do atraso por chegada tardia da aeronave evidencia o efeito cascata: a incapacidade do sistema de se recuperar de atrasos iniciais. Somado ao atraso por companhia aérea, vemos que a maior parte dos minutos perdidos deve-se a fatores internos da operação, e não a fatores externos como clima ou segurança. Isso sugere que modelos preditivos baseados no histórico de ocupação e horários dos aeroportos (hubs) podem ser mais eficazes do que previsões puramente meteorológicas.

In [15]:
df['DISTANCE_GROUP'] = pd.cut(
    df['DISTANCE'], bins=[0, 500, 1000, 1500, 2000, 5000], 
    labels=['0-500 milhas', '500-1000', '1000-1500', '1500-2000', '2000+']
)
df_dist = df.groupby('DISTANCE_GROUP')['IS_DELAYED'].mean().reset_index()

fig = px.bar(
    df_dist, 
    x='DISTANCE_GROUP', 
    y='IS_DELAYED',
    title='Taxa de Atraso por Faixa de Distância',
    labels={'DISTANCE_GROUP': 'Distância do Voo', 'IS_DELAYED': 'Taxa de Atraso'},
    color='IS_DELAYED',
    color_continuous_scale='Blues'
)

fig.update_layout(template='plotly_white')
fig.show()

A distância do voo não apresenta uma correlação linear direta com a probabilidade de atraso. A taxa permanece estável em torno de 18-19% para todas as faixas.

Quando ocorrem os atrasos?

In [16]:
df['SCHEDULED_DEPARTURE_HOUR'] = df['SCHEDULED_DEPARTURE'] // 100
df['SCHEDULED_TIME_HOUR'] = df['SCHEDULED_TIME_HOUR'] = df['SCHEDULED_TIME'] / 60.

df_month = df.groupby('MONTH')['IS_DELAYED'].mean().reset_index()
df_dow = df.groupby('DAY_OF_WEEK')['IS_DELAYED'].mean().reset_index()
df_hour = df.groupby('SCHEDULED_DEPARTURE_HOUR')['IS_DELAYED'].mean().reset_index()

fig = make_subplots(
    rows=3, cols=1, 
    subplot_titles=('% Atraso por Mês', '% Atraso por Dia da Semana', '% Atraso por Hora do Dia'),
    vertical_spacing=0.1
)
fig.add_trace(
    go.Bar(x=df_month['MONTH'], y=df_month['IS_DELAYED'], marker_color='skyblue', name='Mês'),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=df_dow['DAY_OF_WEEK'], y=df_dow['IS_DELAYED'], marker_color='salmon', name='Dia Semana'),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=df_hour['SCHEDULED_DEPARTURE_HOUR'], y=df_hour['IS_DELAYED'], mode='lines+markers', name='Hora'),
    row=3, col=1
)
fig.update_layout(
    height=900, 
    title_text='Análise Temporal de Atrasos',
    showlegend=False,
    template='plotly_white'
)
fig.update_xaxes(title_text='Mês', row=1, col=1)
fig.update_xaxes(title_text='Dia da Semana (1=Seg, 7=Dom)', row=2, col=1)
fig.update_xaxes(title_text='Hora de Partida Programada', row=3, col=1)
fig.show()

Identificamos que a probabilidade de atraso não é uniforme: ela é influenciada por temporadas de férias e climas extremos (Junho e Dezembro), pelo fluxo de viagens de negócios (picos na Segunda e Quinta). O fato de a taxa de atraso saltar de menos de 10% às 5h para mais de 25% às 20h sugere que o congestionamento sistêmico e o acúmulo de atrasos de voos anteriores são os principais gargalos da malha aérea. Para a modelagem preditiva, os gráficos sugerem que as variáveis 'Mês', 'Dia da Semana' e 'Hora de Partida Programada' devem servir de variáveis explanatórias. 

**Nota**: identificamos que os dados de Outubro utilizam uma codificação numérica para os aeroportos que não é compatível com a tabela de referência de aeroportos (IATA). Por esse motivo, após o merge e a limpeza de nulos, os dados deste mês foram removidos. Para o Tech Challenge, seguiremos com os 11 meses restantes, o que ainda representa uma base de dados robusta com milhões de registros.

**Quem atrasa mais?**

In [17]:
df_airline = (
    df.groupby('AIRLINE_NAME')['IS_DELAYED']
    .mean()
    .reset_index()
    .sort_values(by='IS_DELAYED', ascending=True)
)

fig = px.bar(
    df_airline, 
    x='IS_DELAYED', 
    y='AIRLINE_NAME',
    orientation='h',
    title='Proporção de Voos Atrasados por Companhia Aérea',
    labels={'IS_DELAYED': 'Taxa de Atraso (0 a 1)', 'AIRLINE_NAME': 'Companhia Aérea'},
    color='IS_DELAYED',
    color_continuous_scale='Viridis'
)
fig.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    template='plotly_white',
    height=600
)
fig.show()

Onde estão os gargalos?

In [18]:
df_gargalo = df.groupby('ORIGIN_AIRPORT_NAME').agg({
    'IS_DELAYED': ['count', 'mean'], 
    'ARRIVAL_DELAY': 'mean'          
}).reset_index()

df_gargalo.columns = ['ORIGIN_AIRPORT_NAME', 'FLIGHT_COUNT', 'DELAY_RATE', 'AVG_DELAY_MIN']
df_gargalo = df_gargalo.sort_values(by='FLIGHT_COUNT', ascending=False).head(50)
df_gargalo['BUBBLE_SIZE'] = df_gargalo['AVG_DELAY_MIN'].clip(lower=0.1)

fig = px.scatter(
    df_gargalo,
    x='FLIGHT_COUNT',
    y='DELAY_RATE',
    size='BUBBLE_SIZE',
    color='DELAY_RATE',
    hover_name='ORIGIN_AIRPORT_NAME',
    hover_data={
        'BUBBLE_SIZE': False, 
        'AVG_DELAY_MIN': ':.2f', 
        'FLIGHT_COUNT': True, 
        'DELAY_RATE': ':.2%'
    },
    title='Voos Vs. Taxa de Atraso',
    labels={
        'FLIGHT_COUNT': 'Voos',
        'DELAY_RATE': 'Taxa de Atraso',
        'AVG_DELAY_MIN': 'Média de Atraso (min)'
    },
    color_continuous_scale='Reds'
)
fig.update_layout(
    template='plotly_white', 
    height=600
)
fig.show()

In [19]:
df_airports_stats = df.groupby('ORIGIN_AIRPORT').agg({
    'IS_DELAYED': ['mean', 'count']
}).reset_index()
df_airports_stats.columns = ['ORIGIN_AIRPORT', 'DELAY_RATE', 'FLIGHT_VOLUME']

kmeans = KMeans(n_clusters=4, random_state=42)
df_airports_stats['AIRPORT_PROFILE'] = kmeans.fit_predict(df_airports_stats[['DELAY_RATE', 'FLIGHT_VOLUME']])

fig = px.scatter(
    df_airports_stats, 
    x='FLIGHT_VOLUME', 
    y='DELAY_RATE',
    color='AIRPORT_PROFILE',
    hover_name='ORIGIN_AIRPORT',
    title='Agrupamento de Aeroportos: Volume vs. Taxa de Atraso',
    labels={'FLIGHT_VOLUME': 'Volume Total de Voos', 'DELAY_RATE': 'Taxa de Atraso (0 a 1)'},
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Safe
)

fig.show()

A análise de agrupamento revelou que o volume de operações é o principal fator de diferenciação entre os aeroportos. Enquanto grandes centros (hubs) mantêm uma taxa de atraso constante entre 15% e 25%, aeroportos de pequeno porte apresentam alta volatilidade, sendo mais suscetíveis a picos de atraso superiores a 40%, provavelmente devido à menor resiliência operacional.

Correlação entre variáveis numéricas

In [20]:

df = df.merge(df_airports_stats[['ORIGIN_AIRPORT', 'AIRPORT_PROFILE']], on='ORIGIN_AIRPORT', how='left')

num_cols = [
    'MONTH', 'DAY_OF_WEEK', 'SCHEDULED_DEPARTURE_HOUR', 'SCHEDULED_TIME_HOUR', 'DISTANCE', 'AIRPORT_PROFILE', 'IS_DELAYED'
]

corr = df[num_cols].corr()

fig = px.imshow(
    corr,
    text_auto='.2f',
    aspect='auto',
    color_continuous_scale='RdBu_r',
    zmin=-1, 
    zmax=1,
    title='Matriz de Correlação',
    labels=dict(color='Correlação')
)
fig.update_layout(
    height=600,
    template='plotly_white'
)
fig.show()

### Modelagem

Feature engineering

In [21]:
features = [
    'MONTH', 'DAY_OF_WEEK', 'AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'SCHEDULED_DEPARTURE_HOUR', 'SCHEDULED_TIME_HOUR', 'DISTANCE'
]

target = 'IS_DELAYED'

df_ml = df[features + [target]].copy()

for col in ['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'AIRLINE']:
    target_mean = df_ml.groupby(col)[target].mean()
    df_ml[col] = df_ml[col].map(target_mean)


df_atrasados = df_ml[df_ml[target] == 1]
df_pontuais = df_ml[df_ml[target] == 0]

n_atrasados = len(df_atrasados)
df_pontuais_bal = df_pontuais.sample(n=n_atrasados, random_state=42)

df_balanced = pd.concat([df_atrasados, df_pontuais_bal])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(df_balanced.shape[0])

1929746


Treinamento

In [22]:
df_sample = df_balanced.sample(n=min(1000000, len(df_balanced)), random_state=42)

X = df_sample.drop(columns=[target])
y = df_sample[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Distribuição das classes:\n{y.value_counts(normalize=True)}')
print(f'Treino: {X_train.shape}, Teste: {X_test.shape}')

Distribuição das classes:
IS_DELAYED
1    0.500075
0    0.499925
Name: proportion, dtype: float64
Treino: (800000, 8), Teste: (200000, 8)


In [23]:
print('Treinando Regressão Logística...')
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)
print('\nResultado: Regressão Logística')
print(classification_report(y_test, y_pred_lr))

Treinando Regressão Logística...

Resultado: Regressão Logística
              precision    recall  f1-score   support

           0       0.60      0.59      0.60     99985
           1       0.60      0.61      0.60    100015

    accuracy                           0.60    200000
   macro avg       0.60      0.60      0.60    200000
weighted avg       0.60      0.60      0.60    200000



In [24]:
print('Treinando XGBoost...')
xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=10,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    random_state=42,
    n_jobs=-1   
)
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
print('\nResultado:')
print(classification_report(y_test, y_pred_xgb))
print(f'ROC-AUC Score: {roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:, 1]):.4f}')

Treinando XGBoost...

Resultado:
              precision    recall  f1-score   support

           0       0.64      0.62      0.63     99985
           1       0.63      0.66      0.65    100015

    accuracy                           0.64    200000
   macro avg       0.64      0.64      0.64    200000
weighted avg       0.64      0.64      0.64    200000

ROC-AUC Score: 0.6910


In [25]:
def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    x = ['Previsto Pontual', 'Previsto Atrasado']
    y = ['Real Pontual', 'Real Atrasado']
    
    fig = ff.create_annotated_heatmap(
        z=cm, 
        x=x, 
        y=y, 
        annotation_text=cm, 
        colorscale='Blues'
    )
    fig.update_layout(
        title=title,
        xaxis_title='Predição',
        yaxis_title='Realidade',
        template='plotly_white'
    )
    return fig

In [26]:
fig_xgb = plot_confusion_matrix(y_test, y_pred_xgb, 'Matriz de Confusão: XGBoost')
fig_xgb.show()

In [27]:
def plot_roc_curve(y_true, y_probs, title):
    fpr, tpr, _ = roc_curve(y_true, y_probs)
    auc_score = roc_auc_score(y_true, y_probs)

    fig = go.Figure()

    fig.add_trace(go.Scatter(
            x=fpr, 
            y=tpr,
            mode='lines',
            name=f'XGBoost (AUC = {auc_score:.4f})',
            line=dict(color='darkblue', width=3)
        )
    )

    fig.add_trace(go.Scatter(
            x=[0, 1], 
            y=[0, 1],
            mode='lines',
            name='Predição Aleatória',
            line=dict(color='red', dash='dash')
        )
    )

    fig.update_layout(
        title=title,
        xaxis_title='Taxa de Falso Positivo (1 - Especificidade)',
        yaxis_title='Taxa de Verdadeiro Positivo (Sensibilidade)',
        height=600,
        template='plotly_white'
    )
    
    fig.show()

    return auc_score

In [28]:
y_probs_xgb = xgb_model.predict_proba(X_test)[:, 1]
plot_roc_curve(y_test, y_probs_xgb, 'Curva ROC')

0.6910322851982265

Que características aumentam a chance de atraso em um voo?

In [29]:
df_importances = pd.DataFrame({
    'Variável': features,
    'Importância': xgb_model.feature_importances_
}).sort_values(by='Importância', ascending=False)

fig = px.bar(
    df_importances,
    x='Importância',
    y='Variável',
    orientation='h',
    title='Importância das Variáveis Explanatórias'
)
fig.show()